# Data Validation and Preprocessing

This notebook prepares the July 2025 Citi Bike trip data for station-level bike-share demand forecasting. The raw trip records are transformed into an hourly demand matrix for the selected stations. The resulting processed data will be used by the LSTM baseline and the Spatio-Temporal Graph Convolutional Network (STGCN) models.

**Input:** Five raw Citi Bike CSV files for July 2025.  
**Output:** A cleaned hourly station-demand dataset and station metadata saved in `data/processed/`.

## 1. Connect Google Drive

Mount Drive to access raw data and save outputs.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Locate Raw Files

Check that all five July 2025 Citi Bike CSV files are available.

In [2]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/bike_share_stgcn/data/raw")

csv_files = sorted(RAW_DIR.glob("202507-citibike-tripdata_*.csv"))

print(f"Folder found: {RAW_DIR.exists()}")
print(f"Number of CSV files found: {len(csv_files)}\n")

for file in csv_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name:<40} {size_mb:,.1f} MB")

Folder found: True
Number of CSV files found: 5

202507-citibike-tripdata_1.csv           185.8 MB
202507-citibike-tripdata_2.csv           186.5 MB
202507-citibike-tripdata_3.csv           186.4 MB
202507-citibike-tripdata_4.csv           185.9 MB
202507-citibike-tripdata_5.csv           184.2 MB


## 3. Preview the Data

Inspect column names and confirm the timestamp and station-data format.

In [3]:
import pandas as pd

preview = pd.read_csv(csv_files[0], nrows=5)

print("Column names:")
print(preview.columns.tolist())

print("\nFirst 5 rows:")
display(preview)

Column names:
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

First 5 rows:


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,BF31E940F7D80958,electric_bike,2025-07-11 18:42:36.430,2025-07-11 18:46:37.195,N 6 St & Bedford Ave,5379.10,Broadway & Berry St,5164.05,40.717452,-73.958509,40.710361,-73.965304,member
1,0DF0EDCFF6452D83,electric_bike,2025-07-12 19:11:05.558,2025-07-12 19:20:05.429,Huron St & Franklin St,5869.04,Broadway & Berry St,5164.05,40.732660,-73.958260,40.710361,-73.965304,member
2,9AFD3BA9E53733E7,electric_bike,2025-07-02 19:38:18.759,2025-07-02 20:10:05.899,Macon St & Howard Ave,4408.07,Stockholm St & Wilson Ave,4824.03,40.684520,-73.920110,40.699304,-73.923044,casual
3,146A0E14AC873008,classic_bike,2025-07-04 22:03:19.382,2025-07-04 22:37:20.104,Clark St & Henry St,4789.03,Marcus Garvey Blvd & Macon St,4278.03,40.697601,-73.993446,40.682601,-73.938037,member
4,54D1DCA4B21AC7C1,electric_bike,2025-07-07 18:20:14.380,2025-07-07 18:36:38.257,N 6 St & Bedford Ave,5379.10,Stockholm St & Wilson Ave,4824.03,40.717452,-73.958509,40.699304,-73.923044,member


## 4. Validate the Dataset

Count trips, check date ranges and missing values, and identify the busiest pickup stations.

In [4]:
from collections import Counter

USE_COLS = [
    "started_at",
    "start_station_id",
    "start_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
]

CHUNK_SIZE = 200_000

summary_rows = []
total_rows = 0
all_start_station_counts = Counter()

for file in csv_files:
    file_rows = 0
    file_missing_start_id = 0
    file_missing_start_name = 0
    file_missing_start_coords = 0
    file_min_time = None
    file_max_time = None

    for chunk in pd.read_csv(
        file,
        usecols=USE_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        chunk["started_at"] = pd.to_datetime(chunk["started_at"], errors="coerce")

        file_rows += len(chunk)
        file_missing_start_id += chunk["start_station_id"].isna().sum()
        file_missing_start_name += chunk["start_station_name"].isna().sum()
        file_missing_start_coords += (
            chunk["start_lat"].isna() | chunk["start_lng"].isna()
        ).sum()

        valid_times = chunk["started_at"].dropna()
        if not valid_times.empty:
            chunk_min = valid_times.min()
            chunk_max = valid_times.max()

            file_min_time = (
                chunk_min
                if file_min_time is None or chunk_min < file_min_time
                else file_min_time
            )
            file_max_time = (
                chunk_max
                if file_max_time is None or chunk_max > file_max_time
                else file_max_time
            )

        valid_station_ids = chunk["start_station_id"].dropna().astype(str)
        all_start_station_counts.update(valid_station_ids)

    total_rows += file_rows

    summary_rows.append({
        "file": file.name,
        "rows": file_rows,
        "first_start_time": file_min_time,
        "last_start_time": file_max_time,
        "missing_start_station_id": file_missing_start_id,
        "missing_start_station_name": file_missing_start_name,
        "missing_start_coordinates": file_missing_start_coords,
    })

validation_summary = pd.DataFrame(summary_rows)

print(f"TOTAL TRIPS: {total_rows:,}")
print(f"UNIQUE START STATIONS: {len(all_start_station_counts):,}\n")

display(validation_summary)

print("Top 10 pickup stations:")
top_10 = pd.DataFrame(
    all_start_station_counts.most_common(10),
    columns=["start_station_id", "number_of_pickups"]
)
display(top_10)

TOTAL TRIPS: 4,988,053
UNIQUE START STATIONS: 2,157



,file,rows,first_start_time,last_start_time,missing_start_station_id,missing_start_station_name,missing_start_coordinates
0,202507-citibike-tripdata_1.csv,1000000,2025-06-29 23:16:34.475,2025-07-14 19:59:48.441,595,595,595
1,202507-citibike-tripdata_2.csv,1000000,2025-06-30 01:16:29.566,2025-07-14 19:59:53.848,469,469,469
2,202507-citibike-tripdata_3.csv,1000000,2025-06-30 09:55:42.970,2025-07-31 23:55:16.783,431,431,431
3,202507-citibike-tripdata_4.csv,1000000,2025-07-14 20:00:14.281,2025-07-31 23:57:27.342,936,936,936
4,202507-citibike-tripdata_5.csv,988053,2025-07-14 20:00:07.107,2025-07-31 23:55:35.086,566,566,566


Top 10 pickup stations:


,start_station_id,number_of_pickups
0,6140.05,16789
1,6233.04,16682
2,5329.03,16332
3,5788.13,14563
4,6492.08,14470
5,6948.10,14385
6,6726.01,14365
7,6912.01,14155
8,6331.01,14000
9,5905.12,13761


### Validation Summary

The raw data contain 4,988,053 trip records and 2,157 unique pickup stations. A small number of observations lack start-station information and will be removed. Since a few records fall outside July 2025, only trips starting between 1 July and 31 July 2025 will be retained for the forecasting dataset.

## 5. Clean July Pickup Records

Keep valid trips that started in July 2025 and have complete pickup-station information.

In [5]:
PROCESSED_DIR = Path("/content/drive/MyDrive/bike_share_stgcn/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PICKUPS_FILE = PROCESSED_DIR / "clean_july_2025_pickups.csv"

CLEAN_COLS = [
    "started_at",
    "start_station_id",
    "start_station_name",
    "start_lat",
    "start_lng",
    "end_station_id",
]

JULY_START = pd.Timestamp("2025-07-01 00:00:00")
AUGUST_START = pd.Timestamp("2025-08-01 00:00:00")

total_before_cleaning = 0
total_after_cleaning = 0
first_chunk = True

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        usecols=CLEAN_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        total_before_cleaning += len(chunk)

        chunk["started_at"] = pd.to_datetime(chunk["started_at"], errors="coerce")

        clean_chunk = chunk.dropna(
            subset=[
                "started_at",
                "start_station_id",
                "start_station_name",
                "start_lat",
                "start_lng",
            ]
        )

        clean_chunk = clean_chunk[
            (clean_chunk["started_at"] >= JULY_START) &
            (clean_chunk["started_at"] < AUGUST_START)
        ].copy()

        clean_chunk["start_station_id"] = clean_chunk["start_station_id"].astype(str)

        clean_chunk.to_csv(
            CLEAN_PICKUPS_FILE,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False
        total_after_cleaning += len(clean_chunk)

print(f"Rows read: {total_before_cleaning:,}")
print(f"Valid July pickup records saved: {total_after_cleaning:,}")
print(f"Saved file: {CLEAN_PICKUPS_FILE}")

Rows read: 4,988,053
Valid July pickup records saved: 4,984,279
Saved file: /content/drive/MyDrive/bike_share_stgcn/data/processed/clean_july_2025_pickups.csv


### Cleaning Result

After filtering for July 2025 trips with complete pickup-station information, 4,984,279 valid records were retained.

## 6. Select Forecasting Stations

Select the 200 most frequently used pickup stations for the forecasting dataset.

In [6]:
TOP_N_STATIONS = 200
STATION_METADATA_FILE = PROCESSED_DIR / "top_200_station_metadata.csv"

station_counts = {}
station_names = {}
station_lat_sum = {}
station_lng_sum = {}

for chunk in pd.read_csv(
    CLEAN_PICKUPS_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    counts = chunk["start_station_id"].value_counts()

    for station_id, count in counts.items():
        station_counts[station_id] = station_counts.get(station_id, 0) + count

    station_info = (
        chunk.groupby("start_station_id")
        .agg(
            station_name=("start_station_name", "first"),
            latitude=("start_lat", "mean"),
            longitude=("start_lng", "mean"),
        )
    )

    for station_id, row in station_info.iterrows():
        if station_id not in station_names:
            station_names[station_id] = row["station_name"]

        n = counts[station_id]
        station_lat_sum[station_id] = station_lat_sum.get(station_id, 0) + row["latitude"] * n
        station_lng_sum[station_id] = station_lng_sum.get(station_id, 0) + row["longitude"] * n

top_station_ids = sorted(
    station_counts,
    key=station_counts.get,
    reverse=True
)[:TOP_N_STATIONS]

station_metadata = pd.DataFrame({
    "station_id": top_station_ids,
    "station_name": [station_names[s] for s in top_station_ids],
    "latitude": [
        station_lat_sum[s] / station_counts[s]
        for s in top_station_ids
    ],
    "longitude": [
        station_lng_sum[s] / station_counts[s]
        for s in top_station_ids
    ],
    "pickup_count": [station_counts[s] for s in top_station_ids],
})

station_metadata.to_csv(STATION_METADATA_FILE, index=False)

print(f"Stations selected: {len(station_metadata)}")
print(f"Total valid pickups at selected stations: {station_metadata['pickup_count'].sum():,}")
print(
    f"Share of all valid pickup records: "
    f"{station_metadata['pickup_count'].sum() / total_after_cleaning:.2%}"
)

display(station_metadata.head(10))
print(f"\nSaved station metadata to: {STATION_METADATA_FILE}")

Stations selected: 200
Total valid pickups at selected stations: 1,755,685
Share of all valid pickup records: 35.22%


,station_id,station_name,latitude,longitude,pickup_count
0,6140.05,W 21 St & 6 Ave,40.741740,-73.994156,16789
1,6233.04,Pier 61 at Chelsea Piers,40.746872,-74.008210,16681
2,5329.03,West St & Chambers St,40.717548,-74.013221,16329
3,5788.13,Lafayette St & E 8 St,40.730207,-73.991026,14563
4,6492.08,9 Ave & W 33 St,40.752568,-73.996765,14468
5,6948.10,Broadway & W 58 St,40.766953,-73.981693,14385
6,6726.01,11 Ave & W 41 St,40.760301,-73.998842,14363
7,6912.01,7 Ave & Central Park South,40.766741,-73.979069,14155
8,6331.01,W 31 St & 7 Ave,40.749156,-73.991600,13996
9,5905.12,Broadway & E 14 St,40.734546,-73.990741,13760



Saved station metadata to: /content/drive/MyDrive/bike_share_stgcn/data/processed/top_200_station_metadata.csv


## 6. Select Forecasting Stations

Select the 200 most frequently used pickup stations for the forecasting dataset.

In [7]:
selected_pickups = station_metadata["pickup_count"].sum()

print(f"Stations selected: {len(station_metadata)}")
print(f"Total valid pickups at selected stations: {selected_pickups:,}")
print(f"Share of all valid pickup records: {selected_pickups / total_after_cleaning:.2%}")
print(f"Lowest pickup count among selected stations: {station_metadata['pickup_count'].min():,}")

Stations selected: 200
Total valid pickups at selected stations: 1,755,685
Share of all valid pickup records: 35.22%
Lowest pickup count among selected stations: 6,244


### Result

The top 200 stations account for 1,755,685 pickups (35.22% of all valid July 2025 records). The least frequently used selected station recorded 6,244 pickups during the month.

## 7. Aggregate Hourly Station Demand

Aggregate pickups at the 200 selected stations into hourly demand counts.

In [8]:
import numpy as np

HOURLY_DEMAND_FILE = PROCESSED_DIR / "hourly_pickup_demand_top_200.csv"
STATION_ORDER_FILE = PROCESSED_DIR / "top_200_station_order.csv"

selected_station_ids = station_metadata["station_id"].astype(str).tolist()

all_hours = pd.date_range(
    start="2025-07-01 00:00:00",
    end="2025-07-31 23:00:00",
    freq="h"
)

hourly_counts = pd.DataFrame(
    0,
    index=all_hours,
    columns=selected_station_ids,
    dtype=np.int32
)

for chunk in pd.read_csv(
    CLEAN_PICKUPS_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    chunk["start_station_id"] = chunk["start_station_id"].astype(str)

    chunk = chunk[
        chunk["start_station_id"].isin(selected_station_ids)
    ].copy()

    chunk["started_at"] = pd.to_datetime(chunk["started_at"])
    chunk["hour"] = chunk["started_at"].dt.floor("h")

    counts = (
        chunk.groupby(["hour", "start_station_id"])
        .size()
        .unstack(fill_value=0)
    )

    hourly_counts.loc[counts.index, counts.columns] += counts.astype(np.int32)

hourly_counts.index.name = "timestamp"

hourly_counts.to_csv(HOURLY_DEMAND_FILE)

station_metadata[
    ["station_id", "station_name", "latitude", "longitude", "pickup_count"]
].to_csv(STATION_ORDER_FILE, index=False)

matrix_total = hourly_counts.to_numpy().sum()

print(f"Hourly demand matrix shape: {hourly_counts.shape}")
print(f"Total pickups in matrix: {matrix_total:,}")
print(f"Expected selected pickups: {selected_pickups:,}")
print(f"Totals match: {matrix_total == selected_pickups}")
print(
    f"Hours with at least one pickup: "
    f"{(hourly_counts.sum(axis=1) > 0).sum():,} / {len(hourly_counts):,}"
)

display(hourly_counts.iloc[:5, :5])

print(f"\nSaved hourly demand matrix to: {HOURLY_DEMAND_FILE}")
print(f"Saved station order to: {STATION_ORDER_FILE}")

Hourly demand matrix shape: (744, 200)
Total pickups in matrix: 1,755,685
Expected selected pickups: 1,755,685
Totals match: True
Hours with at least one pickup: 744 / 744


,6140.05,6233.04,5329.03,5788.13,6492.08
timestamp,,,,,
2025-07-01 00:00:00,1,0,2,3,1
2025-07-01 01:00:00,2,0,2,2,0
2025-07-01 02:00:00,0,1,1,0,0
2025-07-01 03:00:00,0,0,0,0,0
2025-07-01 04:00:00,1,0,0,0,0



Saved hourly demand matrix to: /content/drive/MyDrive/bike_share_stgcn/data/processed/hourly_pickup_demand_top_200.csv
Saved station order to: /content/drive/MyDrive/bike_share_stgcn/data/processed/top_200_station_order.csv


### Result

The hourly demand matrix has 744 time steps and 200 station series. Its total of 1,755,685 pickups matches the selected-station total, confirming that aggregation was completed correctly.

## 8. Inspect Demand Statistics

Summarize hourly demand across the selected station time series.

In [9]:
station_hourly_totals = hourly_counts.sum(axis=0)

print("Hourly demand per station:")
print(f"Minimum total: {station_hourly_totals.min():,}")
print(f"Median total: {station_hourly_totals.median():,.0f}")
print(f"Maximum total: {station_hourly_totals.max():,}")

print(f"\nStations with zero total demand: {(station_hourly_totals == 0).sum()}")

overall_hourly_demand = hourly_counts.sum(axis=1)
print(f"\nNetwork hourly demand:")
print(f"Minimum: {overall_hourly_demand.min():,}")
print(f"Mean: {overall_hourly_demand.mean():,.1f}")
print(f"Maximum: {overall_hourly_demand.max():,}")

display(
    pd.DataFrame({
        "station_id": station_metadata["station_id"].astype(str),
        "station_name": station_metadata["station_name"],
        "monthly_pickups": station_hourly_totals.values,
        "average_pickups_per_hour": station_hourly_totals.values / len(hourly_counts),
    }).head(10)
)

Hourly demand per station:
Minimum total: 6,244
Median total: 8,144
Maximum total: 16,789

Stations with zero total demand: 0

Network hourly demand:
Minimum: 31
Mean: 2,359.8
Maximum: 7,510


,station_id,station_name,monthly_pickups,average_pickups_per_hour
0,6140.05,W 21 St & 6 Ave,16789,22.565860
1,6233.04,Pier 61 at Chelsea Piers,16681,22.420699
2,5329.03,West St & Chambers St,16329,21.947581
3,5788.13,Lafayette St & E 8 St,14563,19.573925
4,6492.08,9 Ave & W 33 St,14468,19.446237
5,6948.10,Broadway & W 58 St,14385,19.334677
6,6726.01,11 Ave & W 41 St,14363,19.305108
7,6912.01,7 Ave & Central Park South,14155,19.025538
8,6331.01,W 31 St & 7 Ave,13996,18.811828
9,5905.12,Broadway & E 14 St,13760,18.494624


### Result

All 200 selected stations have nonzero demand. Monthly station pickups range from 6,244 to 16,789, while network-wide hourly demand ranges from 31 to 7,510 pickups.